In [1]:
import os
import shutil
import random
import torch
import torchaudio
import torchaudio.transforms as tt
import torch.nn.functional as ff

In [2]:
SR = 16000

In [3]:
subset = 'trn'
vctk_root = r"/home/ovistetom/Documents/Databases_Local/VCTK_092"
old_root = r"/home/ovistetom/Documents/Databases_Local/VCTK_092/wav48_silence_trimmed"

In [4]:
speaker_list = [ d for d in os.listdir(old_root) if os.path.isdir(os.path.join(old_root, d)) ]    

In [5]:
# Return index in audio when 0.1*maximum is reached.
def find_start_index(audio, threshold=0.1):
    max_value = audio.max()
    start_index = torch.where(audio > threshold*max_value)[0][0]
    return start_index.item()

In [6]:
fade_len_in_s = 0.25
fade_len = int(SR * fade_len_in_s)
fade_transform = tt.Fade(fade_in_len=fade_len, fade_out_len=fade_len, fade_shape='linear')


In [ ]:
num_speakers = len(speaker_list)
random.shuffle(speaker_list)

# Compute subset sizes.
trn_size = round(0.8*num_speakers)
tst_size = round(0.1*num_speakers)
val_size = round(0.1*num_speakers)

# Split shuffled list into subsets.
trn_subset = speaker_list[:trn_size]
tst_subset = speaker_list[trn_size:trn_size+tst_size]
val_subset = speaker_list[-val_size:]

for subset, subset_name  in [(trn_subset, 'trn'), (tst_subset, 'tst'), (val_subset, 'val')]:
    for speaker in subset:
        speaker_path_src = os.path.join(old_root, speaker)
        speaker_path_dst = os.path.join(vctk_root, 'sliced_vctk', subset_name, speaker)
        os.makedirs(speaker_path_dst, exist_ok=True)        
        for file_name in os.listdir(speaker_path_src):
            # Load audio file and resample to 16kHz.
            file_path_src = os.path.join(speaker_path_src, file_name)
            audio, sr = torchaudio.load(file_path_src, channels_first=True)
            audio = audio[0]
            resample_transform = tt.Resample(orig_freq=sr, new_freq=SR)
            audio = resample_transform(audio)
            # Find beginning of utterance.
            start_index = find_start_index(audio)
            start_index = max(0, start_index - int(0.25*SR))
            # Slice 4s segment from beginning of utterance.
            audio = audio[start_index:]
            audio = audio[:4*SR] if audio.size(0) > 4*SR else ff.pad(audio, (0, 4*SR-audio.size(0)))
            # Normalize and fade.
            audio /= audio.abs().max()
            audio = fade_transform(audio)
            # Save segment.
            file_path_dst = os.path.join(speaker_path_dst, file_name)
            torchaudio.save(file_path_dst.replace('.flac', '.wav'), audio.unsqueeze(0), SR)            
        